# Datan Puhdistus - Eräajo

Tässä muistikirjassa suoritetaan annettujen sääntöjen mukainen datan puhdistus raakadatalle käsittelemällä jokainen `.csv`-tiedosto vuorotellen yksi kerrallaan tyhjentämättä tietokoneen muistia liikaa. Tallennetaan kaikki lopputulokset yhdeksi jatkuvaksi tiedostoksi.

In [2]:
import pandas as pd
import numpy as np
import datetime
import os
from pathlib import Path

## 1. Puhdistusfunktion määrittely
Kootaan kaikki ennaltamäärätyt puhdistussäännöt yhteen funktioon (Vaiheet 1-10), joka ottaa vastaan yhden csv-tiedoston sisältämän datasetin ja palauttaa puhdistetut kohteiden tiedot, sekä viallisiksi havaitut koordinaattitiedot.

In [3]:
def clean_data(df):
    # 1. z arvon poisto
    if 'z' in df.columns:
        df = df.drop(columns=['z'])
        
    # 2. q alle 50 poisto
    df = df[df['q'] >= 50]
    
    # 3. Virheelliset poistettavat koordinaatit
    invalid_coords_mask = (df['x'] < 0) | (df['y'] < 0) | (df['y'] > 5220) | (df['x'] > 10406)
    removed_coords_df = df[invalid_coords_mask].copy()
    df = df[~invalid_coords_mask]

    # 4. Latausasemien suodatus
    # Hotspot A: x: [0, 200], y: [2400, 2600] (Sisäänkäynti/Laturi)
    # Hotspot B: x: [850, 950], y: [3500, 3700] (Alakerran laturi/Liukuportaat)
    hotspot_mask = (
        ((df['x'] >= 0) & (df['x'] <= 200) & (df['y'] >= 2400) & (df['y'] <= 2600)) |
        ((df['x'] >= 850) & (df['x'] <= 950) & (df['y'] >= 3500) & (df['y'] <= 3700))
    )
    df = df[~hotspot_mask]
    
    # 5. Tunnistetaan sunnuntait muista viikonpäivistä ja suodatetaan aukioloaikojen ulkopuoliset liikkeet pois
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_convert('Europe/Helsinki')
    is_sunday = df['timestamp'].dt.weekday == 6
    time_series = df['timestamp'].dt.time
    
    sunday_valid = is_sunday & (time_series >= datetime.time(10, 0)) & (time_series <= datetime.time(20, 0))
    other_valid = (~is_sunday) & (time_series >= datetime.time(8, 0)) & (time_series <= datetime.time(21, 0))
    df = df[sunday_valid | other_valid]
    
    # 6. Sessiohallinta (Sessionization)
    df = df.sort_values(by=['node_id', 'timestamp'])
    df['dt'] = df.groupby('node_id')['timestamp'].diff().dt.total_seconds()
    # Uusi sessio alkaa jos dt > 60s tai dt on NaN
    df['is_new_session'] = (df['dt'] > 60) | (df['dt'].isna())
    df['session_idx'] = df.groupby('node_id')['is_new_session'].cumsum().astype(int)
    df['session_id'] = df['node_id'].astype(str) + "_" + df['session_idx'].astype(str)
    
    # 7. Iteratiivinen nopeussuodatus (vain sessioiden sisällä)
    # Poistetaan ensin mikro-aikavälit (<0.1s)
    df = df[(df['dt'] > 0.1) | (df['dt'].isna())]
    
    for _ in range(2):
        df['dt'] = df.groupby('session_id')['timestamp'].diff().dt.total_seconds()
        df['dx'] = df.groupby('session_id')['x'].diff()
        df['dy'] = df.groupby('session_id')['y'].diff()
        df['speed'] = np.where(df['dt'] > 0, np.sqrt(df['dx']**2 + df['dy']**2) / df['dt'], 0)
        df = df[(df['speed'] <= 166) | (df['speed'].isna())]
    
    # 8. Täyden duplikaattidatan poisto
    df = df.drop_duplicates()
    
    # Siivotaan apusarakkeet
    df = df.drop(columns=['dt', 'is_new_session', 'session_idx', 'dx', 'dy', 'speed'])
    
    return df, removed_coords_df

## 2. Käsittelysilmukka jokaiselle tiedostolle
Haetaan kaikki raw-kansion CSV:t, luetaan ne muistiin yksi kerrallaan ja prosessointifunktion jälkeen lopputulokset kerätään yksiin kohdetiedostoihin (`cleaned_data.csv` ja `removed_coordinates.csv`).

In [4]:
raw_dir = Path('../data/raw')
csv_files = list(raw_dir.glob('**/*.csv'))

if not csv_files:
    raise ValueError(f"Ei CSV tiedostoja kansiossa {raw_dir}")

os.makedirs('../data/processed', exist_ok=True)
cleaned_path = '../data/processed/cleaned_data.csv'
removed_path = '../data/processed/removed_coordinates.csv'

# Tyhjennetään mahdolliset vanhat versiot alta
for p in [cleaned_path, removed_path]:
    if os.path.exists(p): os.remove(p)

total_raw_rows = 0
total_cleaned_rows = 0
total_removed_rows = 0

for idx, file in enumerate(csv_files, 1):
    print(f"Käsitellään tiedosto {idx}/{len(csv_files)}: {file.name}")
    df = pd.read_csv(file)
    total_raw_rows += len(df)
    
    # Suoritetaan puhdistus datalle
    cleaned_df, removed_df = clean_data(df)
    
    total_cleaned_rows += len(cleaned_df)
    total_removed_rows += len(removed_df)
    
    # Tallennetaan tiedostoihin
    cleaned_df.to_csv(cleaned_path, mode='a', index=False, header=not os.path.exists(cleaned_path))
    removed_df.to_csv(removed_path, mode='a', index=False, header=not os.path.exists(removed_path))

print("\n--- KAIKKI TIEDOSTOT KÄSITELTY ---")
print(f"Alkuperäisiä datarivejä yhteensä: {total_raw_rows:,}")
print(f"Puhdistettuja rivejä tallennettu: {total_cleaned_rows:,}")
print(f"Virheellisiä koordinaatteja poistettu: {total_removed_rows:,}")
print(f"Datasta poistettu yhteensä: {((total_raw_rows - total_cleaned_rows) / total_raw_rows) * 100:.1f} %")

Käsitellään tiedosto 1/31: node_54016.csv
Käsitellään tiedosto 2/31: node_53936.csv
Käsitellään tiedosto 3/31: node_51720.csv
Käsitellään tiedosto 4/31: node_53795.csv
Käsitellään tiedosto 5/31: node_42787.csv
Käsitellään tiedosto 6/31: node_52008.csv
Käsitellään tiedosto 7/31: node_3224.csv
Käsitellään tiedosto 8/31: node_52023.csv
Käsitellään tiedosto 9/31: node_53027.csv
Käsitellään tiedosto 10/31: node_52535.csv
Käsitellään tiedosto 11/31: node_45300.csv
Käsitellään tiedosto 12/31: node_53011.csv
Käsitellään tiedosto 13/31: node_51751.csv
Käsitellään tiedosto 14/31: node_64458.csv
Käsitellään tiedosto 15/31: node_52003.csv
Käsitellään tiedosto 16/31: node_51735.csv
Käsitellään tiedosto 17/31: node_53888.csv
Käsitellään tiedosto 18/31: node_53768.csv
Käsitellään tiedosto 19/31: node_51889.csv
Käsitellään tiedosto 20/31: node_51866.csv
Käsitellään tiedosto 21/31: node_51719.csv
Käsitellään tiedosto 22/31: node_52099.csv
Käsitellään tiedosto 23/31: node_3240.csv
Käsitellään tiedosto 2